# Empirical BOLD PhiID (AAL90)

This notebook ports the old EEG PhiID workflow onto the current empirical fMRI BOLD dataset used in `TVBToolkit`.

Audited lineage:

- legacy MATLAB generator: `/Users/borjan/code/python/TVBEmergence/test/matlab/emergence_measures.m`
- legacy plotting notebook: `/Users/borjan/code/python/AnesthesiaProjectEmergence/phiid_plot.ipynb`
- current BOLD loader: `scripts/brain_states_new_doc_bold_audited.py`
- repo audit note: `docs/phiid_empirical_bold_audit.md`

The old workflow computed all 16 atoms with `PhiIDFull(..., 1, 'idep_xtb')`, but the final figures primarily used the synergy (`sts`) and redundancy (`rtr`) matrices. This notebook keeps that structure while using the audited AAL90 FC reorder already established for the current DoC BOLD data.

In [1]:
from pathlib import Path
import sys
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from tvbtoolkit.analysis import (
    PRIMARY_ATOMS,
    average_atom_matrices_by_group,
    build_matlab_batch_command,
    export_phiid_subject_inputs,
    load_phiid_index,
    plot_group_average_grid,
    save_group_average_outputs,
)
from brain_states_new_doc_bold_audited import (
    _maybe_apply_roi_reordering,
    build_roi_order_reference,
    load_new_doc_subjects,
    resolve_roi_order_names,
    validate_final_roi_order_or_raise,
)

ROOT

PosixPath('/Users/borjan/CNRS/projects/TVBToolkit')

In [ ]:
DATA_ROOT = ROOT / 'data' / 'doc_patients_new_data'
ROI_REORDER_MODE = 'aal90_fc'
REDUNDANCY = 'ccs'
TR_SECONDS = 2.4
EXPORT_STANDARDIZE = None  # use 'demean' or 'zscore' only if you want a transformed variant
MAX_TIMEPOINTS = None      # keep full matched BOLD length by default

OUT_ROOT = ROOT / 'results' / 'phiid_empirical_bold'
INPUT_DIR = OUT_ROOT / 'inputs'
OUTPUT_DIR = OUT_ROOT / 'phiid' / REDUNDANCY
AVG_DIR = OUT_ROOT / 'averages' / REDUNDANCY
FIG_DIR = OUT_ROOT / 'figures' / REDUNDANCY
RUN_MATLAB = True

MATLAB_BIN = '/Applications/MATLAB_R2023b.app/bin/matlab'
MATLAB_TOOLBOX_ROOT = Path('/Users/borjan/code/matlab/elph')
MATLAB_RUNNER = ROOT / 'scripts' / 'phiid_empirical_bold_aal90.m'

for path in (OUT_ROOT, INPUT_DIR, OUTPUT_DIR, AVG_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Missing dataset root: {DATA_ROOT}')

print({'data_root': str(DATA_ROOT), 'output_root': str(OUT_ROOT), 'redundancy': REDUNDANCY})

{'data_root': '/Users/borjan/CNRS/projects/TVBToolkit/data/doc_patients_new_data', 'output_root': '/Users/borjan/CNRS/projects/TVBToolkit/results/phiid_empirical_bold', 'redundancy': 'ccs'}


## 1. Load audited subject-level BOLD and export MATLAB inputs

This reuses the same subject loader and the same FC-side AAL90 reorder (`aal90_fc`) already validated for the empirical DoC dataset in this repo. Each exported `.mat` file stores one subject as `time_series` with shape `90 x T`, matching the historical MATLAB loop that indexed `time_series(row, :)`.

In [3]:
records, load_qc = load_new_doc_subjects(DATA_ROOT)
records_use, reorder_qc, reorder_decision = _maybe_apply_roi_reordering(records, mode=ROI_REORDER_MODE)
roi_ref = build_roi_order_reference(DATA_ROOT)
validate_final_roi_order_or_raise(roi_ref, applied_mode=reorder_decision['applied_mode'])
roi_labels, _ = resolve_roi_order_names(roi_ref, mode=reorder_decision['applied_mode'])

manifest = export_phiid_subject_inputs(
    records_use,
    INPUT_DIR,
    roi_labels=roi_labels,
    max_timepoints=MAX_TIMEPOINTS,
    standardize=EXPORT_STANDARDIZE,
    tr_seconds=TR_SECONDS,
)

print({'n_subjects': int(manifest.shape[0]), 'roi_reorder_mode': reorder_decision['applied_mode']})
display(manifest.head())

{'n_subjects': 179, 'roi_reorder_mode': 'aal90_fc'}


,subject_id,subject_stub,cohort,stage,sedation,n_timepoints,n_regions,source_fc_file,source_sc_file,source_subject_index,source_subject_label,dropped_nonfinite_timepoints
0,control_control_non_sedated_S01,control_control_non_sedated_S01,control,control,non_sedated,297,90,CNT_send/FC/DoC_CNT.mat,CNT_send/SC/CNT_SC.mat,0,S01,0
1,control_control_non_sedated_S02,control_control_non_sedated_S02,control,control,non_sedated,297,90,CNT_send/FC/DoC_CNT.mat,CNT_send/SC/CNT_SC.mat,1,S02,0
2,control_control_non_sedated_S03,control_control_non_sedated_S03,control,control,non_sedated,297,90,CNT_send/FC/DoC_CNT.mat,CNT_send/SC/CNT_SC.mat,2,S03,0
3,control_control_non_sedated_S04,control_control_non_sedated_S04,control,control,non_sedated,297,90,CNT_send/FC/DoC_CNT.mat,CNT_send/SC/CNT_SC.mat,3,S04,0
4,control_control_non_sedated_S05,control_control_non_sedated_S05,control,control,non_sedated,297,90,CNT_send/FC/DoC_CNT.mat,CNT_send/SC/CNT_SC.mat,4,S05,0


## 2. Run the MATLAB PhiID batch

The batch runner mirrors the old EEG script:

- loop over every ordered ROI pair
- call `PhiIDFull([roi_i; roi_j], 1, 'idep_xtb')`
- save one file per atom per subject
- also save the `sr_gradient`

Leave `RUN_MATLAB = False` the first time so you can inspect the export manifest and output paths. Set it to `True` only when you are ready to launch the full job.

In [4]:
matlab_cmd = build_matlab_batch_command(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    redundancy=REDUNDANCY,
    matlab_bin=MATLAB_BIN,
    matlab_toolbox_root=MATLAB_TOOLBOX_ROOT,
    runner_path=MATLAB_RUNNER,
)
print(matlab_cmd)

if RUN_MATLAB:
    completed = subprocess.run(matlab_cmd, shell=True, cwd=ROOT, check=True)
    print({'returncode': completed.returncode})
else:
    print('MATLAB run skipped. Set RUN_MATLAB = True to launch the full PhiID batch.')

/Applications/MATLAB_R2023b.app/bin/matlab -batch "addpath(genpath('/Users/borjan/code/matlab/elph')); addpath('/Users/borjan/CNRS/projects/TVBToolkit/scripts'); phiid_empirical_bold_aal90('/Users/borjan/CNRS/projects/TVBToolkit/results/phiid_empirical_bold/inputs', '/Users/borjan/CNRS/projects/TVBToolkit/results/phiid_empirical_bold/phiid/ccs', 'ccs', true, 0)"
MATLAB run skipped. Set RUN_MATLAB = True to launch the full PhiID batch.


## 3. Reload outputs and average by cohort

This follows the same high-level idea as the old `phiid_plot.ipynb`, but the grouping key here is the current empirical BOLD metadata. The primary outputs are the cohort-average `sts` and `rtr` matrices.

In [5]:
index_df = load_phiid_index(OUTPUT_DIR, manifest_path=INPUT_DIR / 'manifest.csv')
if index_df.empty:
    print('No PhiID outputs found yet. Run the MATLAB batch first.')
else:
    display(index_df.head())
    cohort_avgs = pd.concat(
        [
            average_atom_matrices_by_group(index_df, atom=atom, group_cols=['cohort'])
            for atom in PRIMARY_ATOMS
        ],
        ignore_index=True,
    )
    saved = save_group_average_outputs(cohort_avgs, AVG_DIR / 'by_cohort')
    display(saved)

No PhiID outputs found yet. Run the MATLAB batch first.


In [6]:
if 'cohort_avgs' not in globals() or cohort_avgs.empty:
    print('No averaged outputs available yet.')
else:
    for atom in PRIMARY_ATOMS:
        fig, _ = plot_group_average_grid(
            cohort_avgs,
            atom=atom,
            title_cols=['cohort'],
            roi_labels=roi_labels,
            ncols=3,
        )
        fig.savefig(FIG_DIR / f'cohort_{atom}_grid.png', dpi=220, bbox_inches='tight')
        fig.savefig(FIG_DIR / f'cohort_{atom}_grid.svg', bbox_inches='tight')
        plt.show()

    cohort_stage_avgs = pd.concat(
        [
            average_atom_matrices_by_group(index_df, atom=atom, group_cols=['cohort', 'stage', 'sedation'])
            for atom in PRIMARY_ATOMS
        ],
        ignore_index=True,
    )
    save_group_average_outputs(cohort_stage_avgs, AVG_DIR / 'by_cohort_stage_sedation')
    print({'cohort_groups': int(cohort_avgs.shape[0]), 'cohort_stage_groups': int(cohort_stage_avgs.shape[0])})

No averaged outputs available yet.
